In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import norm
import matplotlib.pyplot as plt

print("Imports successful")

Imports successful


In [2]:
# Cell 2 — ILS Core Geometry & Constants
# All values per ICAO Annex 10 standards

# ILS parameters
GLIDE_SLOPE_ANGLE = 3.0        # degrees (standard ILS glide slope)
LOCALIZER_WIDTH   = 5.0        # degrees full width at threshold
RUNWAY_LENGTH     = 3000       # meters
THRESHOLD_DIST    = 300        # meters from runway start to threshold

# DDM thresholds (Difference in Depth of Modulation)
DDM_FULL_SCALE    = 0.155      # DDM at full-scale deflection
DDM_THRESHOLD     = 0.0        # DDM = 0 means on centerline

# Approach parameters
APPROACH_DIST_NM  = 10         # nautical miles final approach
NM_TO_METERS      = 1852       # 1 nautical mile in meters
APPROACH_DIST_M   = APPROACH_DIST_NM * NM_TO_METERS

# Glide slope intercept altitude
# At 3 degrees, altitude = distance * tan(angle)
def glide_slope_altitude(distance_m):
    """Returns ideal altitude in meters at given distance from threshold"""
    return distance_m * np.tan(np.radians(GLIDE_SLOPE_ANGLE))

# Test at key distances
for d_nm in [10, 5, 2, 1, 0.5]:
    d_m = d_nm * NM_TO_METERS
    alt = glide_slope_altitude(d_m)
    print(f"  {d_nm} NM from threshold → ideal altitude: {alt:.0f} m ({alt*3.281:.0f} ft)")

  10 NM from threshold → ideal altitude: 971 m (3185 ft)
  5 NM from threshold → ideal altitude: 485 m (1592 ft)
  2 NM from threshold → ideal altitude: 194 m (637 ft)
  1 NM from threshold → ideal altitude: 97 m (318 ft)
  0.5 NM from threshold → ideal altitude: 49 m (159 ft)


In [3]:
# Cell 3 — Simulate Aircraft on ILS Approach
np.random.seed(42)
n_points = 500

# Distance array: 10 NM down to threshold
distances = np.linspace(APPROACH_DIST_M, 0, n_points)

# Ideal glide slope altitude
ideal_altitude = glide_slope_altitude(distances)

# Simulate aircraft with realistic deviations
# Crosswind effect + turbulence + pilot corrections
crosswind      = 15          # knots
turbulence_std = 0.8         # meters of random deviation

np.random.seed(42)
noise          = np.random.normal(0, turbulence_std, n_points)
drift          = np.sin(distances / 2000) * crosswind * 0.1

actual_altitude = ideal_altitude + noise + drift

# Glide slope deviation in dots (1 dot = 0.0875 DDM)
gs_deviation_m   = actual_altitude - ideal_altitude
gs_deviation_dots = gs_deviation_m / (distances * np.tan(np.radians(0.35)) + 1e-9)
gs_deviation_dots = np.clip(gs_deviation_dots, -2.5, 2.5)

# Localizer deviation (crosstrack error)
ideal_track      = np.zeros(n_points)
crosstrack_error = np.random.normal(0, 0.3, n_points) + np.sin(distances / 3000) * 2
loc_deviation_dots = np.clip(crosstrack_error / 2, -2.5, 2.5)

print(f"Approach simulated: {n_points} data points")
print(f"Distance range: {distances[-1]:.0f}m to {distances[0]/NM_TO_METERS:.1f} NM")
print(f"\nGlide Slope deviation stats:")
print(f"  Max above GS: +{gs_deviation_dots.max():.3f} dots")
print(f"  Max below GS: {gs_deviation_dots.min():.3f} dots")
print(f"  RMS deviation: {np.sqrt(np.mean(gs_deviation_dots**2)):.4f} dots")

Approach simulated: 500 data points
Distance range: 0m to 10.0 NM

Glide Slope deviation stats:
  Max above GS: +1.197 dots
  Max below GS: -2.500 dots
  RMS deviation: 0.1952 dots


In [4]:
# Cell 4 — Plot 3D ILS Approach Path
distances_nm = distances / NM_TO_METERS

fig = go.Figure()

# Ideal glide slope
fig.add_trace(go.Scatter3d(
    x=distances_nm,
    y=ideal_track,
    z=ideal_altitude,
    mode='lines',
    line=dict(color='lime', width=4),
    name='Ideal Glide Slope'
))

# Actual aircraft path
fig.add_trace(go.Scatter3d(
    x=distances_nm,
    y=crosstrack_error,
    z=actual_altitude,
    mode='lines',
    line=dict(color='crimson', width=2),
    name='Actual Aircraft Path'
))

# Runway
fig.add_trace(go.Scatter3d(
    x=[0, -RUNWAY_LENGTH/NM_TO_METERS],
    y=[0, 0],
    z=[0, 0],
    mode='lines',
    line=dict(color='white', width=6),
    name='Runway'
))

fig.update_layout(
    title='3D ILS Approach — Glide Slope vs Actual Aircraft Path',
    scene=dict(
        xaxis_title='Distance from Threshold (NM)',
        yaxis_title='Lateral Deviation (m)',
        zaxis_title='Altitude (m)',
        bgcolor='rgb(10,10,30)'
    ),
    template='plotly_dark',
    width=900, height=600
)
fig.show()

In [5]:
# Cell 5 — CDI/GS Needle Deviation Display (Cockpit View)
fig = go.Figure()

x_nm = distances_nm

# Glide slope deviation trace
fig.add_trace(go.Scatter(
    x=x_nm,
    y=gs_deviation_dots,
    mode='lines',
    name='GS Deviation (dots)',
    line=dict(color='cyan', width=2)
))

# Localizer deviation trace
fig.add_trace(go.Scatter(
    x=x_nm,
    y=loc_deviation_dots,
    mode='lines',
    name='LOC Deviation (dots)',
    line=dict(color='orange', width=2)
))

# Warning thresholds
for level, color, label in [(1.0, 'yellow', '1 dot'), (2.0, 'red', '2 dots (full scale)')]:
    fig.add_hline(y=level,  line=dict(color=color, dash='dash', width=1), annotation_text=label)
    fig.add_hline(y=-level, line=dict(color=color, dash='dash', width=1))

fig.add_hline(y=0, line=dict(color='white', dash='dot', width=1),
              annotation_text='Centerline')

fig.update_layout(
    title='ILS CDI & Glide Slope Needle Deviation — Final Approach',
    xaxis_title='Distance from Threshold (NM)',
    yaxis_title='Deviation (dots)',
    xaxis=dict(autorange='reversed'),
    template='plotly_dark',
    width=900, height=500
)
fig.show()

In [6]:
# Cell 6 — Decision Height & Autoland Assessment
DECISION_HEIGHT_M = 60   # CAT I ILS: 200ft = ~60m
DECISION_HEIGHT_FT = 200

# Find decision height crossing point
dh_idx = np.argmin(np.abs(ideal_altitude - DECISION_HEIGHT_M))
dh_distance_nm = distances_nm[dh_idx]

gs_at_dh  = gs_deviation_dots[dh_idx]
loc_at_dh = loc_deviation_dots[dh_idx]

# CAT I limits: within 1 dot on both axes
gs_pass  = abs(gs_at_dh)  <= 1.0
loc_pass = abs(loc_at_dh) <= 1.0
land_ok  = gs_pass and loc_pass

print(f"=== ILS Approach Assessment ===")
print(f"\nDecision Height: {DECISION_HEIGHT_FT} ft ({DECISION_HEIGHT_M} m)")
print(f"DH reached at:   {dh_distance_nm:.3f} NM from threshold")
print(f"\nAt Decision Height:")
print(f"  GS  deviation: {gs_at_dh:+.4f} dots  → {'PASS' if gs_pass  else 'FAIL'}")
print(f"  LOC deviation: {loc_at_dh:+.4f} dots  → {'PASS' if loc_pass else 'FAIL'}")
print(f"\nLanding clearance (CAT I): {'LAND' if land_ok else 'GO AROUND'}")

# Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-04-ils-glide-path\outputs'
import os
os.makedirs(output_dir, exist_ok=True)

approach_df = pd.DataFrame({
    'distance_nm':       distances_nm,
    'ideal_altitude_m':  ideal_altitude,
    'actual_altitude_m': actual_altitude,
    'gs_deviation_dots': gs_deviation_dots,
    'loc_deviation_dots':loc_deviation_dots
})
approach_df.to_csv(f'{output_dir}\\approach_data.csv', index=False)
print(f"\nData exported to outputs/")

=== ILS Approach Assessment ===

Decision Height: 200 ft (60 m)
DH reached at:   0.621 NM from threshold

At Decision Height:
  GS  deviation: +0.0557 dots  → PASS
  LOC deviation: +0.3682 dots  → PASS

Landing clearance (CAT I): LAND

Data exported to outputs/
